# cAPTure: XGB-P development training

This CPU-only notebook trains the packet-only XGBoost baseline under the two frozen benign-background folds. It uses completed canonical packet and fold-local preprocessing artifacts on Drive. Each development packet receives exactly one out-of-fold (OOF) score. No test scenario is read, no threshold is selected, and no final five-scenario model is trained here.

Run sections 1–6 for the primary depth-5 configuration. Section 7 is the optional, more expensive depth-10 capacity sensitivity. A completed fold is verified and reused if the same run ID is resumed; an incomplete output is never overwritten automatically.

## 1. Mount Drive and load the current repository

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_WORK_ROOT = Path("/content/capture_xgb_p_work")

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/utils/capture_xgb_p.py", "code/python/tests/test_capture_xgb_p.py", "code/python/requirements-capture-xgb.txt", "configs/capture_experiment_v1.yaml", "configs/capture_packet_schema_v1.yaml", "configs/capture_preprocessing_v1.yaml"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
print("CPU XGB-P environment is ready.")
print("Repository commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())

Mounted at /content/drive
CPU XGB-P environment is ready.
Repository commit: 8c9527db19677f8385730662d0c487dc02c16117


## 2. Run synthetic protocol checks

In [2]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
for pattern in ("test_capture_preprocess.py", "test_capture_xgb_p.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT_ROOT / "code/python/tests"), "-p", pattern, "-v"], env=test_environment, cwd=PROJECT_ROOT, check=True)
print("Synthetic preprocessing and XGB-P checks passed.")

Synthetic preprocessing and XGB-P checks passed.


## 3. Bind the approved development artifacts and run ID

Set `RUN_ID` to an earlier XGB-P run ID when resuming after a Colab disconnect. A new ID starts a new run; old results are never overwritten.

In [3]:
from IPython.display import display
import pandas as pd
from utils.capture_data import load_manifest
from utils.capture_xgb_p import run_xgb_p_fold, summarize_xgb_p_oof, validate_xgb_p_fold_run

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / "20260917T235058_827743Z_prepare_full_dev"
PREPROCESSING_AUDIT_DIR = DRIVE_ROOT / "preprocessing_runs" / "20260919T004143_161396Z_preprocessing"
RUN_ID = None  # Replace with a previous XGB-P run ID only when resuming.
RUN_DEPTH10_SENSITIVITY = False  # Decide from CPU budget before inspecting model metrics.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_xgb_p"
DRIVE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs" / RUN_ID
BATCH_SIZE = 50_000
CPU_THREADS = 2

for path in (PREPARED_RUN_DIR, PREPROCESSING_AUDIT_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required Drive run is missing: {path}")
manifest = load_manifest(MANIFEST_PATH)
assert manifest["training"]["hyperparameter_search"]["xgb_p"]["primary_configuration"] == "depth5_primary"
print("Prepared input:", PREPARED_RUN_DIR)
print("Preprocessing audit:", PREPROCESSING_AUDIT_DIR)
print("XGB-P run ID:", RUN_ID)
print("XGB-P Drive output:", DRIVE_RUN_DIR)
print("Depth-10 sensitivity planned:", RUN_DEPTH10_SENSITIVITY)

Prepared input: /content/drive/MyDrive/capture_gate0/prepared_runs/20260917T235058_827743Z_prepare_full_dev
Preprocessing audit: /content/drive/MyDrive/capture_gate0/preprocessing_runs/20260919T004143_161396Z_preprocessing
XGB-P run ID: 20260919T151844_852400Z_xgb_p
XGB-P Drive output: /content/drive/MyDrive/capture_gate0/xgb_p_runs/20260919T151844_852400Z_xgb_p
Depth-10 sensitivity planned: False


## 4. Check local storage before training

The large fold-B feature matrix is temporary and stays on Colab local storage. Model files, fold reports, and OOF packet scores are copied to Drive and verified. The runner rechecks source hashes, preprocessing provenance, row counts, and free local storage.

In [4]:
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)
local_free_gib = shutil.disk_usage(LOCAL_WORK_ROOT).free / (1024 ** 3)
drive_free_gib = shutil.disk_usage(DRIVE_ROOT).free / (1024 ** 3)
training_rows = {fold: sum(json.loads((PREPARED_RUN_DIR / scenario / "preparation_report.json").read_text(encoding="utf-8"))["counts"]["packets"] for scenario in split["train"]) for fold, split in manifest["validation"]["folds"].items()}
display(pd.DataFrame([{"fold": fold, "training_packets": rows, "estimated_feature_matrix_gib": rows * 103 * 4 / (1024 ** 3)} for fold, rows in training_rows.items()]))
print(f"Free local storage: {local_free_gib:.1f} GiB")
print(f"Free Drive storage: {drive_free_gib:.1f} GiB")
if local_free_gib < max(rows * 103 * 4 / (1024 ** 3) for rows in training_rows.values()) + 2:
    raise OSError("Local storage is too small for the fold-B feature matrix.")

,fold,training_packets,estimated_feature_matrix_gib
0,A,2675496,1.026601
1,B,11533878,4.425606


Free local storage: 87.3 GiB
Free Drive storage: 83.0 GiB


## 5. Train the primary depth-5 configuration

Each fold can take a long time on Colab CPU, especially fold B. Run fold A first, inspect its report, then run fold B. An existing complete fold is checksum-verified instead of retrained.

In [5]:
def train_or_verify(fold, configuration_name):
    output_dir = DRIVE_RUN_DIR / configuration_name / f"fold_{fold}"
    if output_dir.exists():
        report = validate_xgb_p_fold_run(output_dir, fold, configuration_name)
        print(f"Verified existing {configuration_name} fold {fold}: {output_dir}")
        return report
    return run_xgb_p_fold(manifest_path=MANIFEST_PATH, packet_schema_path=PACKET_SCHEMA_PATH, preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH, prepared_run_dir=PREPARED_RUN_DIR, preprocessing_audit_dir=PREPROCESSING_AUDIT_DIR, output_dir=output_dir, local_work_root=LOCAL_WORK_ROOT, fold=fold, configuration_name=configuration_name, batch_size=BATCH_SIZE, nthread=CPU_THREADS)

primary_a = train_or_verify("A", "depth5_primary")
display(pd.DataFrame.from_dict(primary_a["validation"], orient="index")[["rows", "normal_packets", "attack_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

Materializing fold A training matrix on local storage...
Training depth5_primary on 2,675,496 packets with 103 features...
Scoring held-out scenario train_dollar_char...
Scoring held-out scenario train_slash_char...
Scoring held-out scenario train_sub_exf...
Saved depth5_primary fold A to /content/drive/MyDrive/capture_gate0/xgb_p_runs/20260919T151844_852400Z_xgb_p/depth5_primary/fold_A.
Removed temporary local training matrix after verified Drive copy.


,rows,normal_packets,attack_packets,packet_roc_auc,packet_pr_auc_diagnostic
train_dollar_char,2882555,1705003,1177552,0.918163,0.922244
train_slash_char,6425516,1705003,4720513,0.990194,0.996614
train_sub_exf,2225807,1705003,520804,0.983434,0.961152


In [6]:
 imary_b = train_or_verify("B", "depth5_primary")
display(pd.DataFrame.from_dict(primary_b["validation"], orient="index")[["rows", "normal_packets", "attack_packets", "packet_roc_auc", "packet_pr_auc_diagnostic"]])

Materializing fold B training matrix on local storage...
Training depth5_primary on 11,533,878 packets with 103 features...
Scoring held-out scenario train_empty_conn...
Scoring held-out scenario train_qos_mid...
Saved depth5_primary fold B to /content/drive/MyDrive/capture_gate0/xgb_p_runs/20260919T151844_852400Z_xgb_p/depth5_primary/fold_B.
Removed temporary local training matrix after verified Drive copy.


,rows,normal_packets,attack_packets,packet_roc_auc,packet_pr_auc_diagnostic
train_empty_conn,1175779,746806,428973,0.990654,0.982734
train_qos_mid,1499717,746806,752911,0.960903,0.971557


## 6. Review the complete primary OOF result

In [7]:
primary_summary = summarize_xgb_p_oof(DRIVE_RUN_DIR, "depth5_primary")
display(pd.DataFrame.from_dict(primary_summary["scenario_metrics"], orient="index"))
print("Fold ROC-AUC:", primary_summary["fold_packet_roc_auc"])
print("Hierarchical macro OOF packet ROC-AUC:", primary_summary["hierarchical_macro_oof_packet_roc_auc"])
print("Thresholds selected:", primary_summary["thresholds_selected"])

,fold,packets,packet_roc_auc,packet_pr_auc_diagnostic
train_dollar_char,A,2882555,0.918163,0.922244
train_slash_char,A,6425516,0.990194,0.996614
train_sub_exf,A,2225807,0.983434,0.961152
train_empty_conn,B,1175779,0.990654,0.982734
train_qos_mid,B,1499717,0.960903,0.971557


Fold ROC-AUC: {'A': 0.9639302829172967, 'B': 0.9757783128792075}
Hierarchical macro OOF packet ROC-AUC: 0.9698542978982521
Thresholds selected: False


## 6A. Inspect primary artifacts and threshold-free diagnostics

  Verify the saved fold artifacts, inspect feature-importance rankings, and examine score separation by scenario and attack step. These are exploratory diagnostics: they do not select a threshold, change the primary configuration, or access test data.


In [9]:
import gc
import pyarrow.parquet as pq
import numpy as np

artifact_rows = []
importance_rows = []
score_rows = []
attack_step_rows = []

for fold in ("A", "B"):
    fold_dir = DRIVE_RUN_DIR / "depth5_primary" / f"fold_{fold}"
    report = validate_xgb_p_fold_run(fold_dir, fold, "depth5_primary")

    forbidden_features = {
        "binary_label", "attack_step", "phase", "sequence_id",
        "packet_id", "source_row_id", "packet_timestamp_ns",
        "window_index", "window_end_ns",
    }
    if forbidden_features.intersection(report["feature_names"]):
        raise ValueError("Evaluation metadata appeared among model features.")

    gains = json.loads(
        (fold_dir / "feature_importance_gain.json").read_text(encoding="utf-8")
    )
    if set(gains) != set(report["feature_names"]):
        raise ValueError("Feature-importance names do not match the fold report.")

    importance_rows.extend(
        {"fold": fold, "feature": feature, "gain": gain}
        for feature, gain in gains.items()
    )

    for scenario, saved in report["validation"].items():
        oof_path = fold_dir / saved["oof_artifact"]
        rows = pq.ParquetFile(oof_path).metadata.num_rows
        if rows != saved["rows"]:
            raise ValueError(f"OOF row count mismatch for {scenario}.")

        artifact_rows.append({
            "fold": fold,
            "scenario": scenario,
            "oof_rows": rows,
            "model_sha256": report["model_sha256"],
            "oof_sha256": saved["oof_sha256"],
        })

        frame = pq.read_table(
            oof_path, columns=["binary_label", "score", "attack_step"]
        ).to_pandas()
        labels = frame["binary_label"].to_numpy()
        scores = frame["score"].to_numpy(dtype=np.float32, copy=False)

        for label, class_name in ((0, "normal"), (1, "attack")):
            class_scores = scores[labels == label]
            q05, median, q95 = np.quantile(class_scores, [0.05, 0.50, 0.95])
            score_rows.append({
                "fold": fold,
                "scenario": scenario,
                "class": class_name,
                "packets": len(class_scores),
                "score_q05": q05,
                "score_median": median,
                "score_q95": q95,
            })

        normal_scores = np.sort(scores[labels == 0])
        attack_frame = frame.loc[labels == 1, ["attack_step", "score"]]

        for attack_step, group in attack_frame.groupby(
            "attack_step", dropna=False
        ):
            step_scores = group["score"].to_numpy(dtype=np.float32)
            lower = np.searchsorted(normal_scores, step_scores, side="left")
            upper = np.searchsorted(normal_scores, step_scores, side="right")
            step_auc = np.mean((lower + upper) / (2 * len(normal_scores)))

            attack_step_rows.append({
                "fold": fold,
                "scenario": scenario,
                "attack_step": attack_step,
                "attack_packets": len(step_scores),
                "roc_auc_vs_scenario_normal": float(step_auc),
            })

        del frame
        gc.collect()

print("Verified OOF artifacts")
display(pd.DataFrame(artifact_rows)[["fold", "scenario", "oof_rows"]])

print("Top 15 gain-ranked features per fold")
importance_table = pd.DataFrame(importance_rows)
display(
    importance_table.sort_values(
        ["fold", "gain"], ascending=[True, False]
    ).groupby("fold").head(15).reset_index(drop=True)
)

print("OOF score distributions by scenario and class")
display(pd.DataFrame(score_rows))

print("Attack steps with the weakest ranking against normal packets")
display(
    pd.DataFrame(attack_step_rows)
    .sort_values("roc_auc_vs_scenario_normal")
    .head(20)
    .reset_index(drop=True)
)


Verified OOF artifacts


,fold,scenario,oof_rows
0,A,train_dollar_char,2882555
1,A,train_slash_char,6425516
2,A,train_sub_exf,2225807
3,B,train_empty_conn,1175779
4,B,train_qos_mid,1499717


Top 15 gain-ranked features per fold


,fold,feature,gain
0,A,is_mqtt,47014.226562
1,A,tcp_destination_port_role_mqtt_messaging,31049.599609
2,A,frame_length,28337.425781
3,A,is_arp,16726.673828
4,A,ipv4_length,16657.599609
5,A,tcp_flag_ack,12226.843750
6,A,mqtt_message_type_1,8676.674805
7,A,mqtt_duplicate_flag,7264.218262
8,A,mqtt_message_type_2,7208.148438
9,A,tcp_flag_fin,4929.728027


OOF score distributions by scenario and class


,fold,scenario,class,packets,score_q05,score_median,score_q95
0,A,train_dollar_char,normal,1705003,0.000134,0.007468,0.716730
1,A,train_dollar_char,attack,1177552,0.005105,0.996946,0.999993
2,A,train_slash_char,normal,1705003,0.000134,0.007468,0.716730
3,A,train_slash_char,attack,4720513,0.524646,0.999984,0.999993
4,A,train_sub_exf,normal,1705003,0.000134,0.007468,0.716730
5,A,train_sub_exf,attack,520804,0.490722,0.999933,0.999993
6,B,train_empty_conn,normal,746806,0.000053,0.000307,0.492299
7,B,train_empty_conn,attack,428973,0.492299,0.999977,0.999992
8,B,train_qos_mid,normal,746806,0.000053,0.000307,0.492299
9,B,train_qos_mid,attack,752911,0.018776,0.999975,0.999991


Attack steps with the weakest ranking against normal packets


,fold,scenario,attack_step,attack_packets,roc_auc_vs_scenario_normal
0,A,train_sub_exf,mqtt_cat,2550,0.804928
1,A,train_sub_exf,scp_exf,5764,0.861514
2,A,train_dollar_char,dollar_char,739943,0.871215
3,A,train_slash_char,slash_char,389562,0.890376
4,B,train_qos_mid,qos_mid_ddos,257148,0.893205
5,B,train_qos_mid,qos_mid,12477,0.898496
6,B,train_empty_conn,empty_conn_ddos,28269,0.941987
7,B,train_empty_conn,empty_conn,6650,0.945503
8,B,train_empty_conn,sftp_inst,607,0.956609
9,A,train_sub_exf,nmap_mqtt,2852,0.958057


## 7. Optional depth-10 capacity sensitivity

Decide whether the CPU and memory budget permits this sensitivity before inspecting the primary validation metrics. The section can be executed later for practical reasons, but its activation must not depend on model performance. Depth 10 is one of the values in the authors' XGBoost grid. It is not an automatic replacement for the predeclared depth-5 primary comparison. Keep both fold runs under the same run ID.

In [ ]:
if RUN_DEPTH10_SENSITIVITY:
    sensitivity_a = train_or_verify("A", "depth10_sensitivity")
    sensitivity_b = train_or_verify("B", "depth10_sensitivity")
    sensitivity_summary = summarize_xgb_p_oof(DRIVE_RUN_DIR, "depth10_sensitivity")
    display(pd.DataFrame([{"configuration": "depth5_primary", "macro_oof_packet_roc_auc": primary_summary["hierarchical_macro_oof_packet_roc_auc"]}, {"configuration": "depth10_sensitivity", "macro_oof_packet_roc_auc": sensitivity_summary["hierarchical_macro_oof_packet_roc_auc"]}]))
else:
    print("Depth-10 sensitivity is disabled; the primary result remains complete.")